# OpenGeoSys 6 — Workshop Exercise 1

**Goal:** Run steady-state diffusion in OGS, load the VTU results and analyze them with PyVista, VTUinterface and Matplotlib. Optionally compare the same calculation with **PardisoLU**.

Prerequisites: complete `bash install` and `bash install.sh --smoke` successfully. Select the **Python (OGS Workshop)** kernel.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pyvista as pv
import vtuIO
import ogstools as ot
from ogs6py import ogs

import os\nroot = Path(os.environ.get('OGS_WORKSHOP_HOME', str(Path.home() / 'ogs-workshop-test' / 'environment')))
results = root / 'results' / 'cases'
print('OGS tools:', getattr(ot, '__version__', 'loaded'))
print('Results:', results)
assert results.is_dir(), 'Run bash install.sh --smoke first'

## 1. Locate actual OGS results

These files were generated by the automatic numerical reference test. We use the **last VTU output step** of the diffusion case.

In [ ]:
output = results / 'steady_diffusion'
vtus = sorted(output.glob('*.vtu'))
assert vtus, f'No VTU output in {output}'
file = vtus[-1]
mesh = pv.read(file)
print('File:', file.name)
print('Nodes:', mesh.n_points, 'Cells:', mesh.n_cells)
print('Point fields:', list(mesh.point_data))
assert mesh.n_points > 0 and mesh.n_cells > 0

## 2. PyVista and VTUinterface

Select an existing scalar point field and verify that both libraries return consistent data.

In [ ]:
fields = [name for name in mesh.point_data if np.asarray(mesh[name]).ndim == 1]
assert fields, 'No scalar point field found'
field = fields[0]
values_pv = np.asarray(mesh.point_data[field])
io = vtuIO.VTUIO(str(file), dim=2)
values_vtu = np.asarray(io.get_point_field(field))
print('Field:', field, 'Shape:', values_pv.shape)
print('Maximum difference:', float(np.max(np.abs(values_pv-values_vtu))))
assert np.allclose(values_pv, values_vtu, rtol=1e-12, atol=1e-12)

## 3. Plot the results with Matplotlib

This plot is intentionally simple: field values scattered against the x coordinate. For an actual profile, use PyVista's `sample_over_line` later.

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.scatter(mesh.points[:,0], values_pv, s=20)
ax.set(xlabel='x [m]', ylabel=field, title='OGS: steady-state diffusion')
ax.grid(alpha=0.25)
plot_path = root / 'results' / 'workshop_diffusion.png'
fig.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)

## 4. Optional PARDISO comparison

If the additional OGS PardisoLU case has run, compare the numerical output fields of SparseLU and PardisoLU. Report the observed difference without claiming unsupported exact equivalence.

In [ ]:
pardiso_files = sorted((results/'pardiso_steady_diffusion').glob('*.vtu'))
if pardiso_files:
    pardiso = pv.read(pardiso_files[-1])
    vals2 = np.asarray(pardiso.point_data[field])
    assert vals2.shape == values_pv.shape
    print('SparseLU versus PardisoLU: max |Δ| =', float(np.max(np.abs(vals2-values_pv))))
else:
    print('PardisoLU test results are missing. Run bash install.sh --smoke.')

## 5. What has actually been tested?

- OGS executed the reference case and wrote result files.
- PyVista and VTUinterface read the same field consistently.
- Matplotlib produces a plot.
- `ogs6py` was imported; XML project generation is tested separately by `verify.py`.

Other process cases (Small Deformation, Richards Flow, Hydro-Mechanics, TRM and TH2M) have separate result folders and can be analyzed in the same way.